# Build 2 — creative retrieval runs on the **Build 1 Lakebase Search index** (not a separate store)

**Execution proof** that the Campaign Desk app's `search_creatives` tool retrieves from the Build 1
Lakebase Search **BM25 index** (`app.creatives_search_bm25` over `app.creatives_search`) — the same index
built in Build 1 — and **not** any external / separate search store. This notebook connects to the same
Lakebase Postgres database and runs the app's **verbatim** retrieval query, then shows the `EXPLAIN` plan
proving the query is served *by that index*. Export it **with cell outputs** as the committed evidence.

App source (verbatim): `src/server/db/queries/campaigns.ts` → `searchCreatives()`, wired to the agent tool
`search_creatives` in `src/server/agent/campaigndesk.ts`. The query below is byte-for-byte the SQL that
function issues (only the `:q` / `:limit` bind params are filled in).


In [1]:
%pip install --quiet psycopg2-binary


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import psycopg2, pandas as pd
from databricks.sdk import WorkspaceClient

HOST     = "ep-delicate-mountain-d8kizg3e.database.us-east-2.cloud.databricks.com"
DBNAME   = "brightwave_lakebase_tkop"
ENDPOINT = "projects/brightwave-campaign-desk/branches/production/endpoints/primary"

w = WorkspaceClient()
cred = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": ENDPOINT})
conn = psycopg2.connect(host=HOST, dbname=DBNAME, user=w.current_user.me().user_name,
                        password=cred["token"], sslmode="require")
conn.set_session(readonly=True)

def run(sql, params=None):
    with conn.cursor() as cur:
        # Only pass params when present — otherwise psycopg2 tries to %-interpolate
        # literal '%' in the SQL (e.g. ILIKE '%bm25%') and errors.
        if params:
            cur.execute(sql, params)
        else:
            cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print("Connected to Lakebase Postgres:", run("SELECT current_database()").iloc[0, 0])


Connected to Lakebase Postgres: brightwave_lakebase_tkop


## 1. The Build 1 Lakebase Search index exists (`lakebase_bm25` over `app.creatives_search`)


In [3]:
idx = run("""
  SELECT n.nspname AS schema, t.relname AS table_name, c.relname AS index_name, am.amname AS index_type
    FROM pg_class c
    JOIN pg_am am ON am.oid = c.relam
    JOIN pg_index i ON i.indexrelid = c.oid
    JOIN pg_class t ON t.oid = i.indrelid
    JOIN pg_namespace n ON n.oid = t.relnamespace
   WHERE n.nspname = 'app' AND c.relname = 'creatives_search_bm25'
""")
print(idx.to_string(index=False))
n = run("SELECT COUNT(*) AS indexed_creatives FROM app.creatives_search").iloc[0, 0]
print(f"\nindexed creatives in app.creatives_search: {n}")


schema       table_name            index_name    index_type
   app creatives_search creatives_search_bm25 lakebase_bm25

indexed creatives in app.creatives_search: 400


## 2. The app's VERBATIM retrieval query (BM25 over the Build 1 index) — ranked hits
The exact SQL from `searchCreatives()`: `search_tsv <@> to_bm25query(to_tsvector('english', :q), 'app.creatives_search_bm25'::regclass)`, `ORDER BY … ASC`.


In [4]:
Q = "testimonial social apparel gen_z"   # a real search_creatives query (:q); :limit = 6

# ↓↓↓ byte-for-byte the SQL issued by src/server/db/queries/campaigns.ts → searchCreatives()
APP_QUERY = """
  SELECT creative_id, creative_name, creative_type, angle, description,
         (search_tsv <@> to_bm25query(to_tsvector('english', %(q)s), 'app.creatives_search_bm25'::regclass)) AS score
  FROM app.creatives_search
  ORDER BY search_tsv <@> to_bm25query(to_tsvector('english', %(q)s), 'app.creatives_search_bm25'::regclass) ASC
  LIMIT %(limit)s
"""
hits = run(APP_QUERY, {"q": Q, "limit": 6})
print(f"query :q = {Q!r}\n")
print(hits[["creative_id", "creative_name", "creative_type", "score"]].to_string(index=False))


query :q = 'testimonial social apparel gen_z'

creative_id            creative_name creative_type     score
  CRE-00034     testimonial image 34         image -2.491628
  CRE-00318     testimonial copy 318          copy -2.491628
  CRE-00048     testimonial image 48         image -2.491628
  CRE-00247 testimonial carousel 247      carousel -2.491628
  CRE-00338     testimonial copy 338          copy -2.491628
  CRE-00129    testimonial video 129         video -2.491628


## 3. `EXPLAIN` — retrieval is served BY the Build 1 index (not a seq scan / separate store)


In [5]:
plan = run("EXPLAIN " + APP_QUERY, {"q": Q, "limit": 6})
for line in plan.iloc[:, 0].tolist():
    print(line)

served_by_index = any("creatives_search_bm25" in str(l) and "Index Scan" in str(l) for l in plan.iloc[:, 0])
print(f"\nServed by the Build 1 lakebase_bm25 index (Index Scan using creatives_search_bm25): {served_by_index}")


Limit  (cost=0.00..3.62 rows=6 width=150)
  ->  Index Scan using creatives_search_bm25 on creatives_search  (cost=0.00..241.00 rows=400 width=150)
        Order By: (search_tsv <@> '("''apparel'':3 ''gen'':4 ''social'':2 ''testimoni'':1 ''z'':5",app.creatives_search_bm25)'::bm25query_tsvector)

Served by the Build 1 lakebase_bm25 index (Index Scan using creatives_search_bm25): True


## 4. There is NO separate creative-search store — the app schema's only search index IS the Build 1 index


In [6]:
app_indexes = run("""
  SELECT tablename, indexname, indexdef
    FROM pg_indexes
   WHERE schemaname = 'app' AND (indexdef ILIKE '%bm25%' OR indexdef ILIKE '%ann%' OR tablename = 'creatives_search')
   ORDER BY tablename, indexname
""")
print(app_indexes.to_string(index=False))
print("\nThe app's search_creatives tool binds to 'app.creatives_search_bm25'::regclass — the Build 1 index above.")
print("No external vector DB / separate search service is involved; retrieval is 100% Lakebase Search.")


       tablename             indexname                                                                                                   indexdef
creatives_search  creatives_search_ann CREATE INDEX creatives_search_ann ON app.creatives_search USING lakebase_ann (embedding vector_cosine_ops)
creatives_search creatives_search_bm25                CREATE INDEX creatives_search_bm25 ON app.creatives_search USING lakebase_bm25 (search_tsv)
creatives_search creatives_search_pkey                CREATE UNIQUE INDEX creatives_search_pkey ON app.creatives_search USING btree (creative_id)

The app's search_creatives tool binds to 'app.creatives_search_bm25'::regclass — the Build 1 index above.
No external vector DB / separate search service is involved; retrieval is 100% Lakebase Search.


---
**Result:** the Build 1 Lakebase Search index (`app.creatives_search_bm25`, type `lakebase_bm25`, over 400
creatives) exists; the app's verbatim `searchCreatives()` query returns BM25-ranked hits; and `EXPLAIN` shows
`Index Scan using creatives_search_bm25` — so the Campaign Desk retrieves creatives **from the Build 1
Lakebase Search index, not a separate store**. These executed cell outputs are the committed proof.


In [7]:
conn.close()
